In [6]:
from pathlib import Path
import sys

import polars as pl


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config.py").exists() and (candidate / "services").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from services.dataset_service import DatasetService
from services.preprocessing_service import PreprocessingService

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_cell_alignment("LEFT")
pl.Config.set_fmt_str_lengths(10_000)
pl.Config.set_tbl_width_chars(10_000)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(25)

polars.config.Config

In [7]:
DATASET_PATH = config.DATASETS / "tokopedia-product-reviews-2019.csv"
OUTPUT_DATASET_PATH = config.OUTPUTS / "datasets" / "tokopedia_calibration.csv"
OUTPUT_SUMMARY_PATH = config.ARTIFACTS / "tokopedia_calibration_summary.json"

RANDOM_SEED = 42
SOURCE_NAME = "tokopedia"


print(f"Dataset: {DATASET_PATH}")

Dataset: E:\School\tugas-akhir\project\datasets\tokopedia-product-reviews-2019.csv


In [8]:
dataset_service = DatasetService()
preprocessing_service = PreprocessingService()

raw_df = dataset_service.load(DATASET_PATH)
if config.COL_TEXT not in raw_df.columns:
    raise KeyError(f"Kolom '{config.COL_TEXT}' tidak ditemukan pada dataset")

text_df = (
    raw_df
    .select(pl.col(config.COL_TEXT))
    .with_row_index(config.COL_ID, offset=1)
    .with_columns(
        pl.col(config.COL_ID).cast(pl.Utf8),
        pl.lit(SOURCE_NAME).alias(config.COL_SOURCE),
        pl.col(config.COL_TEXT).cast(pl.Utf8, strict=False),
    )
    .select(config.COL_ID, config.COL_SOURCE, config.COL_TEXT)
)

print(f"Baris sumber: {raw_df.height:,}")
print(f"Kolom yang digunakan: {text_df.columns}")
text_df

Baris sumber: 40,607
Kolom yang digunakan: ['id', 'source', 'text']


id,source,text
"""1""","""tokopedia""","""Barang sesuai pesanan dan cepat sampai"""
"""2""","""tokopedia""","""Barang bagus harga murah"""
"""3""","""tokopedia""","""Paket rapi...mantap....cepat....sampe ke tujuan"""
"""4""","""tokopedia""","""ya saya puas dgn barangnya"""
"""5""","""tokopedia""","""Responya luar biasa b mantap"""
"""6""","""tokopedia""","""seller top, pengiriman cepat barang oke"""
"""7""","""tokopedia""","""pengiriman cepat seller top"""
"""8""","""tokopedia""","""Produk sesuai dengan spec di web dan respon seller sangat cepat. Thankyou."""
"""9""","""tokopedia""","""Respon super cepat, pengiriman cepat, Barang bagus sesuai deskripsi penjual..thx gan"""
"""10""","""tokopedia""","""Barang mantap, pelayanan cepat"""


In [9]:
initial_validation = dataset_service.validate(text_df)
initial_summary = dataset_service.build_summary(text_df)

print("Validasi awal:")
print(initial_validation)
print("Ringkasan awal:")
print(initial_summary)

clean_df = text_df.filter(
    pl.col(config.COL_TEXT).is_not_null()
    & (pl.col(config.COL_TEXT).str.strip_chars().str.len_chars() > 0)
)
clean_df = dataset_service.deduplicate(clean_df)

clean_validation = dataset_service.validate(clean_df)
if not clean_validation["is_valid"]:
    raise ValueError(f"Dataset tidak valid setelah pembersihan: {clean_validation['issues']}")

print(f"Baris setelah pembersihan dan deduplikasi: {clean_df.height:,}")
print(f"Baris terbuang: {text_df.height - clean_df.height:,}")

Validasi awal:
{'total_rows': 40607, 'required_columns': ['id', 'source', 'text'], 'missing_columns': [], 'null_text': 0, 'empty_text': 0, 'empty_source': 0, 'is_valid': True, 'issues': []}
Ringkasan awal:
{'total_data': 40607, 'distribusi_sumber': {'tokopedia': 40607}, 'jumlah_data_kosong': 0, 'jumlah_duplikat': 3306, 'rata_rata_panjang_teks': 55.37}
Baris setelah pembersihan dan deduplikasi: 37,301
Baris terbuang: 3,306


In [10]:
processed_df = preprocessing_service.process_dataframe(clean_df)
processed_df = processed_df.filter(
    pl.col(config.COL_PROCESSED).is_not_null()
    & (pl.col(config.COL_PROCESSED).str.strip_chars().str.len_chars() > 0)
)

print(f"Baris setelah preprocessing: {processed_df.height:,}")
processed_df.select(
    config.COL_ID,
    config.COL_TEXT,
    config.COL_PROCESSED,
)

Baris setelah preprocessing: 37,259


id,text,processed_text
"""1""","""Barang sesuai pesanan dan cepat sampai""","""barang sesuai pesanan dan cepat sampai"""
"""2""","""Barang bagus harga murah""","""barang bagus harga murah"""
"""3""","""Paket rapi...mantap....cepat....sampe ke tujuan""","""paket rapi.mantap.cepat.sampe ke tujuan"""
"""4""","""ya saya puas dgn barangnya""","""ya saya puas dengan barangnya"""
"""5""","""Responya luar biasa b mantap""","""responya luar biasa b mantap"""
"""6""","""seller top, pengiriman cepat barang oke""","""seller top, pengiriman cepat barang oke"""
"""7""","""pengiriman cepat seller top""","""pengiriman cepat seller top"""
"""8""","""Produk sesuai dengan spec di web dan respon seller sangat cepat. Thankyou.""","""produk sesuai dengan spec di web dan respon seller sangat cepat. thankyou."""
"""9""","""Respon super cepat, pengiriman cepat, Barang bagus sesuai deskripsi penjual..thx gan""","""respon super cepat, pengiriman cepat, barang bagus sesuai deskripsi penjual.terima kasih gan"""
"""10""","""Barang mantap, pelayanan cepat""","""barang mantap, pelayanan cepat"""
